In [2]:
import pandas as pd
import openai
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

### read sample dataset

In [11]:
df_items = pd.read_json('../../data/raw/meta_Electronics_2022_2023_ratings_100_sample_1000.jsonl', lines=True)


In [12]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Computers,Galaxy Tab E Lite 7.0 Case - Leafbook Slim Lig...,4.5,153,[Specifically Designed For: Samsung Galaxy Tab...,[Specifically design for Samsung Galaxy Tab E ...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Leafbook,"[Electronics, Computers & Accessories, Tablet ...","{'Standing screen display size': '7 Inches', '...",B01DJI7TP8,NaN,NaN,NaN
1,All Electronics,Lume Cube AIR Video Conference Lighting Kit - ...,3.7,215,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],LUME CUBE,"[Electronics, Camera & Photo, Lighting & Studi...",{'Product Dimensions': '1.61 x 1.61 x 1.1 inch...,B07M6SWKLT,NaN,NaN,NaN
2,All Electronics,Ultimate Ears BOOM 2 Portable Waterproof & Sho...,4.5,1278,[Insanely good 360˚ sound - Ultimate Ears Boom...,[Ultimate Ears boom 2 is the 360-degree wirele...,68.49,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'HONEST review of the UE Boom 2 spe...,Ultimate Ears,"[Electronics, Portable Audio & Video, Portable...",{'Product Dimensions': '2.75 x 2.75 x 7.08 inc...,B07J31M7PW,NaN,NaN,NaN
3,All Electronics,Seagate FreeAgent GoFlex TV STAJ100,3.5,108,[Watch streaming media directly from the Inter...,"[Product Description, The Seagate GoFlex TV HD...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Seagate,"[Electronics, Television & Video, Streaming Me...","{'Brand Name': 'Seagate', 'Item Weight': '10.7...",B003IT6YFK,NaN,NaN,NaN
4,All Electronics,"JSAUX RCA Cable（10ft/3M, RCA Audio Cable Shiel...",4.5,110,[🎧【High-quality Stereo Sound】Built-in multiple...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'J&D 2RCA Male to 2RCA Male Cable',...",JSAUX,"[Electronics, Television & Video, Accessories,...",{'Package Dimensions': '10.16 x 3.15 x 0.71 in...,B099ZPY68J,NaN,NaN,NaN


In [7]:
list(df_items['features'].values)[0]

["Specifically Designed For: Samsung Galaxy Tab 3 Lite 7.0 7'' T110 T111 Tablet / Galaxy Tab E Lite 7.0 (NOT FIT FOR Samsung Galaxy Tab 3 7.0 T210)",
 'Premium PU leather exterior and microfiber interior. Ultra slim and lightweight hard back design adds minimal bulk while protecting your precious device.',
 'Precise cut-outs and openings for easy access to all tablet features.',
 'Cover has flip capability to transform the case into a viewing stand and keyboard stand. One piece case, the front and back does not separate.',
 'Various colors are available.Magnetic closure.']

In [8]:
list(df_items['images'].values)[0]

[{'thumb': 'https://m.media-amazon.com/images/I/51PML408loL._AC_US40_.jpg',
  'large': 'https://m.media-amazon.com/images/I/51PML408loL._AC_.jpg',
  'variant': 'MAIN',
  'hi_res': 'https://m.media-amazon.com/images/I/719i+AMvocL._AC_SL1000_.jpg'},
 {'thumb': 'https://m.media-amazon.com/images/I/51lclDdsfZL._AC_US40_.jpg',
  'large': 'https://m.media-amazon.com/images/I/51lclDdsfZL._AC_.jpg',
  'variant': 'PT01',
  'hi_res': 'https://m.media-amazon.com/images/I/61H-oKcUr0L._AC_SL1000_.jpg'},
 {'thumb': 'https://m.media-amazon.com/images/I/51X5YjAc9mL._AC_US40_.jpg',
  'large': 'https://m.media-amazon.com/images/I/51X5YjAc9mL._AC_.jpg',
  'variant': 'PT02',
  'hi_res': 'https://m.media-amazon.com/images/I/61djvW-muYL._AC_SL1000_.jpg'},
 {'thumb': 'https://m.media-amazon.com/images/I/51TPrYSPW8L._AC_US40_.jpg',
  'large': 'https://m.media-amazon.com/images/I/51TPrYSPW8L._AC_.jpg',
  'variant': 'PT03',
  'hi_res': 'https://m.media-amazon.com/images/I/61kRGWeGHjL._AC_SL1000_.jpg'},
 {'thumb

### preprocess titles and features 
want one single text to descripe the item

In [13]:
def preprocess_item(row):
    return f'{row["title"]} {" ".join(row["features"])}'

def extract_first_large_image(row):
    return row['images'][0].get('large', "")

In [14]:
df_items['preprocessed_description'] = df_items.apply(preprocess_item, axis=1)
df_items['image'] = df_items.apply(extract_first_large_image, axis=1)
df_items.head()


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author,preprocessed_description,image
0,Computers,Galaxy Tab E Lite 7.0 Case - Leafbook Slim Lig...,4.5,153,[Specifically Designed For: Samsung Galaxy Tab...,[Specifically design for Samsung Galaxy Tab E ...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Leafbook,"[Electronics, Computers & Accessories, Tablet ...","{'Standing screen display size': '7 Inches', '...",B01DJI7TP8,NaN,NaN,NaN,Galaxy Tab E Lite 7.0 Case - Leafbook Slim Lig...,https://m.media-amazon.com/images/I/51PML408lo...
1,All Electronics,Lume Cube AIR Video Conference Lighting Kit - ...,3.7,215,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],LUME CUBE,"[Electronics, Camera & Photo, Lighting & Studi...",{'Product Dimensions': '1.61 x 1.61 x 1.1 inch...,B07M6SWKLT,NaN,NaN,NaN,Lume Cube AIR Video Conference Lighting Kit - ...,https://m.media-amazon.com/images/I/411wKstRw1...
2,All Electronics,Ultimate Ears BOOM 2 Portable Waterproof & Sho...,4.5,1278,[Insanely good 360˚ sound - Ultimate Ears Boom...,[Ultimate Ears boom 2 is the 360-degree wirele...,68.49,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'HONEST review of the UE Boom 2 spe...,Ultimate Ears,"[Electronics, Portable Audio & Video, Portable...",{'Product Dimensions': '2.75 x 2.75 x 7.08 inc...,B07J31M7PW,NaN,NaN,NaN,Ultimate Ears BOOM 2 Portable Waterproof & Sho...,https://m.media-amazon.com/images/I/41kYEBxNW2...
3,All Electronics,Seagate FreeAgent GoFlex TV STAJ100,3.5,108,[Watch streaming media directly from the Inter...,"[Product Description, The Seagate GoFlex TV HD...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Seagate,"[Electronics, Television & Video, Streaming Me...","{'Brand Name': 'Seagate', 'Item Weight': '10.7...",B003IT6YFK,NaN,NaN,NaN,Seagate FreeAgent GoFlex TV STAJ100 Watch stre...,https://m.media-amazon.com/images/I/31Vy2ogKvD...
4,All Electronics,"JSAUX RCA Cable（10ft/3M, RCA Audio Cable Shiel...",4.5,110,[🎧【High-quality Stereo Sound】Built-in multiple...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'J&D 2RCA Male to 2RCA Male Cable',...",JSAUX,"[Electronics, Television & Video, Accessories,...",{'Package Dimensions': '10.16 x 3.15 x 0.71 in...,B099ZPY68J,NaN,NaN,NaN,"JSAUX RCA Cable（10ft/3M, RCA Audio Cable Shiel...",https://m.media-amazon.com/images/I/41EbtDHhNE...


### sample 50 items from dataset

In [16]:
df_sample = df_items.sample(50, random_state=42)
df_sample.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author,preprocessed_description,image
521,All Electronics,"Laxihub Indoor Home Security Camera, P2 Baby M...",4.0,6459,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Laxihub P2 1080p Indoor Wireless C...,LAXIHUB,"[Electronics, Camera & Photo, Video Surveillan...",{'Package Dimensions': '6.61 x 5.51 x 3.11 inc...,B08THC8HPD,NaN,NaN,NaN,"Laxihub Indoor Home Security Camera, P2 Baby M...",https://m.media-amazon.com/images/I/51xQLCPORo...
737,Camera & Photo,"Neewer NW-760 HD Monitor, IPS 7"" for Sony Cano...",4.3,454,[High Contrast: 1200:1; High Resolution: Full ...,"[Note:Camera,Tripod and battery are not includ...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Neewer,"[Electronics, Camera & Photo, Lighting & Studi...",{'Product Dimensions': '8.8 x 2.6 x 7.7 inches...,B01LWYDXNF,NaN,NaN,NaN,"Neewer NW-760 HD Monitor, IPS 7"" for Sony Cano...",https://m.media-amazon.com/images/I/51HQM5o0yj...
740,Industrial & Scientific,4 Pack Replacement 3.5mm Male Plug to Bare Wir...,4.4,138,[4 Pack 12in Replacement 3.5mm male plug conne...,[4 Pack Replacement 3.5mm Male Plug to Bare Wi...,5.97,[{'thumb': 'https://m.media-amazon.com/images/...,[],Accessonico,"[Electronics, Home Audio, Home Audio Accessori...","{'Brand': 'Accessonico', 'Connector Type': 'Au...",B08DR4K78X,NaN,NaN,NaN,4 Pack Replacement 3.5mm Male Plug to Bare Wir...,https://m.media-amazon.com/images/I/41rbB+pFJE...
660,All Electronics,Samsung LN40A650 40-Inch 1080p 120Hz LCD HDTV ...,4.1,437,"[Touch of Color Bezel, Full HD 1080p Resolutio...","[Product description, Samsung 40"" 1080p LCD HD...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],SAMSUNG,"[Electronics, Television & Video, Televisions,...","{'Brand Name': 'SAMSUNG', 'Item Weight': '37.5...",B0014175NE,NaN,NaN,NaN,Samsung LN40A650 40-Inch 1080p 120Hz LCD HDTV ...,https://m.media-amazon.com/images/I/312u2KwkRL...
411,All Electronics,"GE UltraPro 6 Outlet Surge Protector, 2 ft Des...",4.7,2301,[Power more - Expand the reach of your outlet ...,[Connect your devices confidently and convenie...,9.88,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': '6 Outlet Surge Protector (Home Off...,GE,"[Electronics, Television & Video, Accessories,...","{'Manufacturer': 'Jasco Products Company, LLC'...",B08DLDCJQR,NaN,NaN,NaN,"GE UltraPro 6 Outlet Surge Protector, 2 ft Des...",https://m.media-amazon.com/images/I/41Bt6VJ6om...


### define embedding function

openai: https://developers.openai.com/api/docs/guides/embeddings

In [18]:
response = openai.embeddings.create(input="Random Text", model='text-embedding-3-small')
len(response.data[0].embedding)

1536

In [24]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(input=text, model=model)
    return response.data[0].embedding


### create qdrant collection

In [21]:
qdrant_client = QdrantClient(url = "http://localhost:6333")

In [22]:
qdrant_client.create_collection(
    collection_name="amazon-electronics-items-collection-01",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

### add data to qdrant


True

### embed data
need to create point struct, to ingest into qdrant

In [ ]:
# example
pointstruct = PointStruct(
    id = 0,
    vector = get_embedding("hello world"),
    payload  = {"text": "hello world", "mdoel": "text-embedding-3-small"}
)

In [29]:
pointstruct

PointStruct(id=0, vector=[-0.006793975830078125, -0.0391845703125, 0.03411865234375, 0.0287322998046875, -0.024810791015625, -0.0419921875, -0.030303955078125, 0.049346923828125, -0.013946533203125, -0.0177001953125, 0.01537322998046875, -0.0269775390625, -0.0209808349609375, -0.027801513671875, 0.0086212158203125, 0.03570556640625, -0.053680419921875, -0.0023174285888671875, 0.008819580078125, 0.048065185546875, 0.037109375, -0.00922393798828125, -0.00875091552734375, 0.01140594482421875, 0.01410675048828125, -0.0021686553955078125, -0.037567138671875, 0.045440673828125, 0.01125335693359375, -0.039642333984375, 0.0234222412109375, -0.05059814453125, 0.01203155517578125, -3.224611282348633e-05, 0.016021728515625, 0.006099700927734375, 0.031951904296875, 0.0034084320068359375, -0.00860595703125, -0.01058197021484375, -0.037384033203125, -0.03448486328125, 0.050048828125, 0.0193634033203125, -0.01432037353515625, 0.0153045654296875, -0.0543212890625, 0.050445556640625, 0.04541015625, 0.0

### amazon data

In [35]:
embed_cols = ['preprocessed_description', 'image', 'rating_number', 'price', 'average_rating', 'parent_asin']
data_to_embed = df_sample[embed_cols].to_dict(orient="records")

In [37]:
point_structs = [PointStruct(
    id = i,
    vector = get_embedding(row["preprocessed_description"]),
    payload = row
) for i, row in enumerate(data_to_embed)]



In [38]:
len(point_structs)

50

### write to qdrant

In [39]:
qdrant_client.upsert(
    collection_name="amazon-electronics-items-collection-01",
    points=point_structs
)


UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

### function for data retrieval

In [42]:
def retrieve_from_qdrant(query, k=5):
    response = qdrant_client.query_points(
        collection_name="amazon-electronics-items-collection-01",
        query=get_embedding(query),
        limit=k
    )
    return response

In [44]:
retrieve_from_qdrant("do you have a usd connectable fan for hot weather?").points
# extra words such as hot weather make it harder for retrieval

[ScoredPoint(id=49, version=1, score=0.38531762, payload={'preprocessed_description': 'ThreeH USB Fan Laptop Cooling Pad with 3 Fans & Blue LED Lights for Laptop PS3 / PS4 / PS Slim H-UF01 Size: Fits 10-15" laptops - Works fine for laptops either smaller or larger than its size. USB port: Powered through USB port,anti-slide. Design: 3 Blue LED lights built into the pad. Good heat dissipation: 3 Fans to avoid overheating,keep your computer good performance. Anti-slide: Round rubber pad with a high friction coefficient on four corner of the fan around to ensure that the notebook will not be damaged by sliding.', 'image': 'https://m.media-amazon.com/images/I/41kKfMW9IYL._AC_.jpg', 'rating_number': 386, 'price': 16.99, 'average_rating': 3.8, 'parent_asin': 'B016KQ7IGG'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=14, version=1, score=0.28863275, payload={'preprocessed_description': 'KMC 6-Outlet Surge Tap, 2 USB Ports (3.4A), 980 Joules Surge Protector, White (2 Pack) 